[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_90_Phase10_Capstone_RAG_Service.ipynb)

# Lesson 90 — Phase 10 Capstone: ship a production `rag-service`
### Phase 10 · RAG at Production Scale — the capstone (closes the phase)

**Where we are.** Across Phase 10 you rebuilt the retrieval stack one piece at a time — and each piece was a *separate notebook*. Today you do the thing that actually matters for a portfolio: **fold all of it into one small, installable, tested, deployable service** with a single `/ask` endpoint.

A stranger should be able to `git clone`, `pip install -e .`, `uvicorn ... /ask`, and `pytest` — and everything works **with no API key**. That constraint (keyless, deterministic, CI-gated) is what turns "a notebook that worked once" into "software someone can trust."

**The whole Phase 10 stack, in one call:**

```
answer(question)
  = chunk (L84)
  → transform the query (L88)
  → hybrid retrieve + RRF (L85)
  → agentic multi-hop loop w/ self-routing (L89)
  → rerank (L86)
  → grounded, cited, abstaining answer (L87)
```

By the end you'll have a real OSS-shaped repo (`rag_service/` package, `tests/` eval gate, `pyproject.toml`, `Dockerfile`, GitHub Actions CI) written to disk, imported, served over HTTP, and passing a quality gate — all offline.

## §1 · From six notebooks to one package

Each Phase-10 lesson becomes exactly one module. That mapping *is* the architecture:

| Lesson | Module | Responsibility | Key primitive |
|---|---|---|---|
| L84 | `corpus.py` | documents → overlapping chunks | `chunk_docs()`, `Chunk` |
| L85 | `retriever.py` | sparse + dense signals, fused | `HybridIndex`, RRF `fuse_rankings()` |
| L88 | `transform.py` | rewrite the query into corpus vocab | `expand_query()`, `multi_query_retrieve()` |
| L86 | `rerank.py` | cross-encoder-style precision reorder | `rerank()` |
| L87 | `grounding.py` | cite evidence, abstain when unsure | `coverage()`, `compose_answer()` |
| L89 | `agent.py` | grade → mine bridge → hop; self-route | `agentic_retrieve()`, `mine_bridge_terms()` |
| — | `llm.py` | real-or-mock model boundary | `call_llm()`, `llm_mode()` |
| — | `pipeline.py` | wire it together | `build_index()`, `answer()` |
| — | `api.py` | FastAPI HTTP shell | `POST /ask` |

**Design rule we keep from L86–L89:** the model is behind one seam (`call_llm`). With a key it's real; without one it's a deterministic mock. Everything downstream — retrieval, grading, the eval gate — is pure Python, so CI is reproducible.

## §0 · Setup

**You do not need an API key.** The service runs on a deterministic mock LLM. If you set `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in Colab Secrets (🔑), the query-rewrite and answer-composition steps use the real model instead — same code path.

The only install is FastAPI (for the HTTP `/ask` demo). If it isn't available the notebook automatically falls back to calling `answer()` in-process, so every cell still runs.

In [ ]:
# One install for the HTTP layer. Everything else is Python stdlib.
# (Pin httpx<0.28 so Starlette's TestClient stays happy on Colab.)
!pip install -q fastapi "httpx<0.28" uvicorn pydantic 2>/dev/null

import os, sys, textwrap, json
BASE = "/content" if os.path.isdir("/content") else os.getcwd()   # Colab vs local
print("Project root:", BASE)
print("Python:", sys.version.split()[0])

## §2 · Write the whole project to disk

A capstone isn't a script — it's a *repo*. This cell materializes the full package tree under `BASE/`: the `rag_service` package, the `tests` eval gate, and the packaging/CI files. Read each module in the sections that follow; here we just lay them down and put `BASE` on `sys.path` so we can import them.

In [ ]:
import os, pathlib
FILES = {}
FILES['rag_service/__init__.py'] = r'''"""rag_service — a compact, production-shaped RAG service.

Phase 10 in one importable package:
  chunk (L84) -> transform (L88) -> hybrid retrieve + RRF (L85)
  -> rerank (L86) -> agentic multi-hop loop w/ self-routing (L89)
  -> grounded, cited, abstaining answer (L87).

Public API:
    from rag_service import answer, build_index
"""
from .pipeline import answer, build_index
from .llm import call_llm, llm_mode

__all__ = ["answer", "build_index", "call_llm", "llm_mode"]
__version__ = "0.1.0"
'''
FILES['rag_service/llm.py'] = r'''"""Real-or-mock LLM boundary (same pattern as L86-L89).

If a provider key is present in the environment (ANTHROPIC_API_KEY or
OPENAI_API_KEY) we call the real model. Otherwise we fall back to a
DETERMINISTIC mock so the whole service — and its CI eval gate — runs
green with zero credentials. The mock is intentionally simple: it only
does the two language jobs the pipeline needs (paraphrase a query,
compose a grounded answer from evidence) using a small domain map.
"""
from __future__ import annotations
import os, re

_SYN = {
    "flat": ["stale beans", "no crema", "under-extracted"],
    "lifeless": ["stale beans", "no crema"],
    "foam": ["crema", "milk froth"],
    "bitter": ["over-extracted", "grind too fine", "water too hot"],
    "sour": ["under-extracted", "grind too coarse", "water too cool"],
    "weak": ["under-extracted", "dose too low"],
    "leak": ["gasket", "seal", "portafilter"],
    "not hot": ["boiler", "thermoblock", "temperature"],
    "cold": ["boiler", "thermoblock", "temperature"],
    "slow": ["boiler", "heat-up time"],
    "clean": ["backflush", "descale", "maintenance"],
}

def llm_mode() -> str:
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "anthropic"
    if os.environ.get("OPENAI_API_KEY"):
        return "openai"
    return "mock"

def _mock(prompt: str, task: str) -> str:
    low = prompt.lower()
    if task == "paraphrase":
        hits = []
        for k, vs in _SYN.items():
            if k in low:
                hits.extend(vs)
        # dedupe, keep order
        seen, out = set(), []
        for h in hits:
            if h not in seen:
                seen.add(h); out.append(h)
        return "; ".join(out[:4])
    if task == "answer":
        # Evidence block is appended after 'EVIDENCE:' — echo it back as a
        # grounded sentence with citations already embedded by the caller.
        m = re.search(r"EVIDENCE:\s*(.*)", prompt, re.S)
        return (m.group(1).strip() if m else "").split("\n")[0][:400]
    return ""

def call_llm(prompt: str, task: str = "answer", max_tokens: int = 300) -> str:
    """task in {'paraphrase','answer'}. Returns text. Never raises on missing key."""
    mode = llm_mode()
    if mode == "mock":
        return _mock(prompt, task)
    try:
        if mode == "anthropic":
            import anthropic
            c = anthropic.Anthropic()
            r = c.messages.create(model="claude-3-5-haiku-latest",
                                  max_tokens=max_tokens,
                                  messages=[{"role": "user", "content": prompt}])
            return r.content[0].text.strip()
        else:
            from openai import OpenAI
            c = OpenAI()
            r = c.chat.completions.create(model="gpt-4o-mini",
                                          max_tokens=max_tokens,
                                          messages=[{"role": "user", "content": prompt}])
            return r.choices[0].message.content.strip()
    except Exception:
        return _mock(prompt, task)
'''
FILES['rag_service/corpus.py'] = r'''"""Corpus + chunking (L84).

A tiny home-espresso knowledge base. Documents are arranged so some facts
live ONE HOP apart (the answer to a question names a term you must look up
in a *different* document) — that is what forces the agentic loop in L89 to
earn its keep. Chunking = sentence windows with overlap so no fact is split.
"""
from __future__ import annotations
import re
from dataclasses import dataclass

DOCS = {
    "d_grind": (
        "Grind size controls extraction. A grind that is too fine over-extracts and "
        "tastes bitter. A grind that is too coarse under-extracts and tastes sour. "
        "The reference machine for these notes is the Silvia Pro."
    ),
    "d_taste": (
        "Sour, thin shots usually mean under-extraction: coarsen the grind or raise "
        "water temperature. Bitter, harsh shots mean over-extraction: go finer or cool "
        "the water. A flat, lifeless shot with no crema points to stale beans."
    ),
    "d_temp": (
        "Brew temperature depends on the boiler. The Silvia Pro uses a dual-boiler "
        "design with PID control, so it holds temperature within one degree. Thermoblock "
        "machines swing more and need a longer warm-up."
    ),
    "d_crema": (
        "Crema is the reddish-brown foam on a good shot. Fresh beans and correct dose "
        "produce thick crema. No crema at all usually means stale beans or a broken seal."
    ),
    "d_leak": (
        "Water leaking from the portafilter points to a worn group gasket. Replacing the "
        "gasket restores the seal. A broken seal can also thin the crema."
    ),
    "d_warm": (
        "The Silvia Pro needs roughly a twenty minute wait after switching on. "
        "Only then does the group reach a stable extraction condition."
    ),
    "d_clean": (
        "Backflush the machine weekly with a blind basket and detergent. Descale the "
        "boiler monthly in hard-water areas to protect the PID temperature probe."
    ),
}

@dataclass(frozen=True)
class Chunk:
    id: str
    doc: str
    text: str

def _sentences(text: str):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

def chunk_docs(docs=DOCS, window: int = 2, overlap: int = 1):
    """Sentence-window chunks with overlap. Deterministic ids: <doc>#<n>."""
    chunks = []
    for doc, text in docs.items():
        sents = _sentences(text)
        step = max(1, window - overlap)
        n = 0
        i = 0
        while i < len(sents):
            win = sents[i:i + window]
            if not win:
                break
            chunks.append(Chunk(id=f"{doc}#{n}", doc=doc, text=" ".join(win)))
            n += 1
            if i + window >= len(sents):
                break
            i += step
    return chunks
'''
FILES['rag_service/retriever.py'] = r'''"""Hybrid retrieval + Reciprocal Rank Fusion (L85).

Two complementary, dependency-free signals over the same chunks:
  * SPARSE — IDF-weighted bag-of-words cosine (great on exact term overlap).
  * DENSE  — character-trigram cosine (a transparent stand-in for embeddings;
             robust to word-form / typo mismatch without any model download).
We fuse their ranked lists with RRF (k=60), which is score-scale-agnostic and
the same fusion we use to merge query variants in transform.py.
"""
from __future__ import annotations
import re, math
from collections import Counter, defaultdict

def _tok(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())

def _trigrams(s: str):
    s = re.sub(r"\s+", " ", s.lower())
    return [s[i:i+3] for i in range(max(0, len(s) - 2))]

class HybridIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        self.ids = [c.id for c in chunks]
        # sparse
        self.tf = [Counter(_tok(c.text)) for c in chunks]
        df = Counter()
        for tf in self.tf:
            df.update(tf.keys())
        N = len(chunks)
        self.idf = {t: math.log((N + 1) / (d + 0.5)) + 1.0 for t, d in df.items()}
        # dense (char trigram)
        self.tg = [Counter(_trigrams(c.text)) for c in chunks]

    def _sparse_scores(self, q):
        qv = Counter(_tok(q))
        out = []
        for tf in self.tf:
            num = sum(qv[t] * tf.get(t, 0) * (self.idf.get(t, 1.0) ** 2) for t in qv)
            dn = math.sqrt(sum((tf.get(t, 0) * self.idf.get(t, 1.0)) ** 2 for t in tf)) or 1.0
            qn = math.sqrt(sum((qv[t] * self.idf.get(t, 1.0)) ** 2 for t in qv)) or 1.0
            out.append(num / (dn * qn))
        return out

    def _dense_scores(self, q):
        qv = Counter(_trigrams(q))
        qn = math.sqrt(sum(v * v for v in qv.values())) or 1.0
        out = []
        for tg in self.tg:
            num = sum(qv[t] * tg.get(t, 0) for t in qv)
            dn = math.sqrt(sum(v * v for v in tg.values())) or 1.0
            out.append(num / (dn * qn))
        return out

    @staticmethod
    def _rank(scores):
        return [i for i, _ in sorted(enumerate(scores), key=lambda x: -x[1])]

    def search(self, query: str, k: int = 5, rrf_k: int = 60):
        s_rank = self._rank(self._sparse_scores(query))
        d_rank = self._rank(self._dense_scores(query))
        fused = defaultdict(float)
        for rank, idx in enumerate(s_rank):
            fused[idx] += 1.0 / (rrf_k + rank + 1)
        for rank, idx in enumerate(d_rank):
            fused[idx] += 1.0 / (rrf_k + rank + 1)
        order = sorted(fused, key=lambda i: -fused[i])[:k]
        return [(self.chunks[i], fused[i]) for i in order]

def fuse_rankings(list_of_id_lists, rrf_k: int = 60):
    """RRF-merge several ranked id lists (used by multi-query transform)."""
    fused = defaultdict(float)
    for ids in list_of_id_lists:
        for rank, cid in enumerate(ids):
            fused[cid] += 1.0 / (rrf_k + rank + 1)
    return sorted(fused, key=lambda c: -fused[c])
'''
FILES['rag_service/transform.py'] = r'''"""Query transformation (L88): fix the query before it hits the index.

Users type everyday words ("flat", "no foam"); the corpus speaks in technical
terms ("stale beans", "crema", "under-extracted"). Multi-Query asks the LLM to
paraphrase the question into the corpus vocabulary; we retrieve for the original
AND each variant, then RRF-fuse the ranked lists (retriever.fuse_rankings).
Keyless, the mock LLM supplies a domain synonym expansion so this still helps.
"""
from __future__ import annotations
from .llm import call_llm

def expand_query(query: str, n: int = 3):
    """Return [original, *variants]. Variants use corpus vocabulary."""
    prompt = (
        "Rewrite this espresso troubleshooting question using precise coffee "
        "terminology (extraction, crema, grind, boiler, gasket). "
        f"Give up to {n} short alternative phrasings.\nQUESTION: {query}"
    )
    raw = call_llm(prompt, task="paraphrase")
    variants = [v.strip() for v in raw.replace("\n", ";").split(";") if v.strip()]
    out = [query]
    for v in variants:
        if v.lower() not in (o.lower() for o in out):
            out.append(f"{query} {v}")  # keep original intent, add vocabulary
    return out[: n + 1]

def multi_query_retrieve(index, query: str, k: int = 5, n: int = 3):
    """Retrieve for original + variants, RRF-fuse to a single chunk list."""
    from .retriever import fuse_rankings
    variants = expand_query(query, n=n)
    ranked_id_lists = []
    id2chunk = {}
    for v in variants:
        hits = index.search(v, k=k)
        ranked_id_lists.append([c.id for c, _ in hits])
        for c, _ in hits:
            id2chunk[c.id] = c
    fused_ids = fuse_rankings(ranked_id_lists)
    return [(id2chunk[cid], 0.0) for cid in fused_ids[:k]], variants
'''
FILES['rag_service/rerank.py'] = r'''"""Cross-encoder-style reranking (L86).

A bi-encoder / hybrid retriever scores query and chunk independently. A
cross-encoder looks at the PAIR together and is far more precise — at a cost.
We emulate that with a transparent pairwise scorer: term coverage (how many
query terms appear) + proximity (are they close together in the chunk?). This
reorders the retriever's top-k so the most on-point chunk floats up.
"""
from __future__ import annotations
import re

def _tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def _pair_score(query, text):
    q = [t for t in _tok(query) if len(t) > 2]
    if not q:
        return 0.0
    toks = _tok(text)
    positions = {t: [i for i, w in enumerate(toks) if w == t] for t in set(q)}
    covered = [t for t in set(q) if positions[t]]
    coverage = len(covered) / len(set(q))
    # proximity: min span covering the matched query terms (smaller = better)
    prox = 0.0
    if len(covered) >= 2:
        firsts = [min(positions[t]) for t in covered]
        lasts = [max(positions[t]) for t in covered]
        span = max(lasts) - min(firsts) + 1
        prox = len(covered) / span
    return coverage + 0.5 * prox

def rerank(query, scored_chunks, top_n=3):
    """scored_chunks: list[(Chunk, retriever_score)] -> top_n [(Chunk, ce_score)]."""
    rescored = [(c, _pair_score(query, c.text)) for c, _ in scored_chunks]
    rescored.sort(key=lambda x: -x[1])
    return rescored[:top_n]
'''
FILES['rag_service/grounding.py'] = r'''"""Grounding, citations & abstention (L87).

Rules the answer step obeys:
  1. Answer ONLY from retrieved evidence (no free-floating model knowledge).
  2. Every claim carries a [chunk_id] citation.
  3. If the evidence does not sufficiently cover the question, ABSTAIN
     ("I don't have enough grounded information ...") instead of guessing.
"""
from __future__ import annotations
import re

STOP = set("a an the is are was were do does did why how what which when to my "
           "of in on with for and or no not it this that shot my me your you i "
           "well good bad have has can should would".split())

def content_terms(query: str):
    return [t for t in re.findall(r"[a-z0-9]+", query.lower())
            if t not in STOP and len(t) > 2]

def coverage(query: str, chunks):
    """Fraction of query content terms present anywhere in the evidence."""
    terms = set(content_terms(query))
    if not terms:
        return 1.0, set()
    blob = " ".join(c.text.lower() for c in chunks)
    present = {t for t in terms if re.search(rf"\b{re.escape(t)}", blob)}
    return len(present) / len(terms), (terms - present)

def compose_answer(query: str, chunks, from_llm):
    """Build a grounded, cited answer string. `from_llm` composes prose from
    an EVIDENCE block; the mock simply grounds on the top sentence."""
    if not chunks:
        return "I don't have enough grounded information to answer that.", []
    cited = []
    lines = []
    for c in chunks:
        lines.append(f"{c.text} [{c.id}]")
        cited.append(c.id)
    evidence = "\n".join(lines)
    prompt = (f"Answer the question using ONLY the evidence. Keep the [chunk_id] "
              f"citations.\nQUESTION: {query}\nEVIDENCE: {evidence}")
    text = from_llm(prompt, task="answer")
    if "[" not in text:                      # ensure a citation always survives
        text = f"{text} [{cited[0]}]"
    return text, cited
'''
FILES['rag_service/agent.py'] = r'''"""Agentic multi-hop loop with self-routing (L89).

Single-shot RAG fails on BRIDGE questions where the question shares no words
with the answer passage but the first hop names a linking term. The loop:

    retrieve -> GRADE evidence -> if answered: stop (self-routing)
             -> else mine BRIDGE terms -> hop (retrieve again, augmented)
             -> repeat until graded-good OR hop budget spent -> answer/abstain

Self-routing falls out for free: an easy question grades 'good' on hop 1 and
terminates immediately; a bridge question keeps hopping until it links.

The GRADE step is an LLM's job in production (does this evidence actually answer
the question?). Keyless, we use a deterministic proxy: coverage of the query's
*corpus-answerable* content terms across accumulated evidence, plus a retrieval
floor so genuinely out-of-domain questions (no lexical footing) always abstain.
"""
from __future__ import annotations
import re
from .grounding import coverage, content_terms, STOP
from .rerank import _pair_score

def _corpus_vocab(index):
    v = set()
    for tf in index.tf:
        v.update(tf.keys())
    return v

def grade(index, query, seen_chunks, threshold):
    """Return (is_good, coverage, best_pair_score, missing_terms)."""
    vocab = _corpus_vocab(index)
    required = [t for t in content_terms(query) if t in vocab]
    chunks = list(seen_chunks)
    best = max((_pair_score(query, c.text) for c in chunks), default=0.0)
    if not required:
        # No corpus-answerable terms in the query -> only ground if some chunk
        # is a strong lexical match; otherwise it's out-of-domain -> abstain.
        return (best >= 1.0, 1.0 if best >= 1.0 else 0.0, best, set())
    blob = " ".join(c.text.lower() for c in chunks)
    present = {t for t in required if re.search(rf"\b{re.escape(t)}", blob)}
    cov = len(present) / len(required)
    good = cov >= threshold and best > 0.0
    return good, cov, best, set(required) - present

def mine_bridge_terms(index, chunks, query, max_terms=3):
    """Specific, recurring terms in current evidence NOT in the query — the
    links to follow next. df>=2 (appears in multiple chunks) + high idf,
    stopwords excluded."""
    qterms = set(content_terms(query))
    cand = {}
    for c in chunks:
        for t in set(re.findall(r"[a-z0-9]+", c.text.lower())):
            if len(t) <= 2 or t in qterms or t in STOP:
                continue
            idf = index.idf.get(t, 0.0)
            df = sum(1 for tf in index.tf if tf.get(t))
            if df >= 2 and idf >= 1.0:
                cand[t] = max(cand.get(t, 0.0), idf * df)
    return [t for t, _ in sorted(cand.items(), key=lambda x: -x[1])[:max_terms]]

def agentic_retrieve(index, query, retrieve_fn, grade_threshold=0.75,
                     max_hops=3, k=5, top_n=3, trace=None):
    """retrieve_fn(index, q) -> (list[(Chunk,score)], variants). Returns the
    final reranked evidence, a hop trace, and whether it graded good."""
    from .rerank import rerank
    trace = trace if trace is not None else []
    q = query
    seen = {}
    evidence = []
    for hop in range(1, max_hops + 1):
        hits, _ = retrieve_fn(index, q)
        for c, s in hits:
            seen[c.id] = c
        pool = [(c, 0.0) for c in seen.values()]
        evidence = rerank(q, pool, top_n=top_n)  # current (augmented) query surfaces bridge hits
        good, cov, best, missing = grade(index, query, seen.values(), grade_threshold)
        trace.append({"hop": hop, "query": q, "coverage": round(cov, 3),
                      "best_score": round(best, 3),
                      "evidence": [c.id for c, _ in evidence],
                      "missing": sorted(missing)})
        if good:
            return evidence, trace, True
        bridges = mine_bridge_terms(index, [c for c, _ in evidence], query)
        if not bridges:
            break
        q = f"{query} {' '.join(bridges)}"
    good, *_ = grade(index, query, seen.values(), grade_threshold)
    return evidence, trace, good
'''
FILES['rag_service/pipeline.py'] = r'''"""The public pipeline: build_index() + answer().

answer() folds the whole of Phase 10 into one call:
  transform (L88) -> hybrid retrieve + RRF (L85) -> agentic multi-hop
  w/ self-routing (L89) -> rerank (L86) -> grounded/cited/abstaining answer (L87).
Chunking (L84) happens once in build_index().
"""
from __future__ import annotations
from dataclasses import dataclass, field
from .corpus import chunk_docs, DOCS
from .retriever import HybridIndex
from .transform import multi_query_retrieve
from .agent import agentic_retrieve
from .grounding import coverage, compose_answer
from .llm import call_llm, llm_mode

def build_index(docs=DOCS, window=2, overlap=1):
    return HybridIndex(chunk_docs(docs, window=window, overlap=overlap))

@dataclass
class Answer:
    query: str
    text: str
    grounded: bool
    citations: list = field(default_factory=list)
    hops: int = 0
    trace: list = field(default_factory=list)
    llm_mode: str = "mock"

    def to_dict(self):
        return {"query": self.query, "answer": self.text, "grounded": self.grounded,
                "citations": self.citations, "hops": self.hops,
                "llm_mode": self.llm_mode, "trace": self.trace}

def answer(query: str, index=None, grade_threshold=0.75, max_hops=3,
           k=5, top_n=3) -> Answer:
    index = index or build_index()
    def retrieve_fn(idx, q):
        return multi_query_retrieve(idx, q, k=k)
    evidence, trace, ok = agentic_retrieve(
        index, query, retrieve_fn, grade_threshold=grade_threshold,
        max_hops=max_hops, k=k, top_n=top_n)
    chunks = [c for c, _ in evidence]
    cov, _ = coverage(query, chunks)
    if not ok:                                   # abstain rather than guess (L87)
        return Answer(query, "I don't have enough grounded information to answer "
                      "that confidently.", False, [], len(trace), trace, llm_mode())
    text, cites = compose_answer(query, chunks, call_llm)
    return Answer(query, text, True, cites, len(trace), trace, llm_mode())
'''
FILES['rag_service/api.py'] = r'''"""FastAPI serving layer — a thin HTTP shell over answer().

The intelligence lives in pipeline.answer(); the API just validates input,
builds the index once at startup, and returns the grounded answer as JSON.
Run locally:  uvicorn rag_service.api:app --reload   then POST /ask.
"""
from __future__ import annotations
from pydantic import BaseModel, Field
from fastapi import FastAPI
from .pipeline import build_index, answer
from .llm import llm_mode

app = FastAPI(title="rag-service", version="0.1.0")
_INDEX = None

def get_index():
    global _INDEX
    if _INDEX is None:
        _INDEX = build_index()
    return _INDEX

class AskRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=500)
    max_hops: int = Field(3, ge=1, le=5)

class AskResponse(BaseModel):
    question: str
    answer: str
    grounded: bool
    citations: list[str]
    hops: int
    llm_mode: str

@app.get("/health")
def health():
    return {"status": "ok", "llm_mode": llm_mode()}

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    a = answer(req.question, index=get_index(), max_hops=req.max_hops)
    return AskResponse(question=a.query, answer=a.text, grounded=a.grounded,
                       citations=a.citations, hops=a.hops, llm_mode=a.llm_mode)
'''
FILES['tests/eval_set.py'] = r'''"""Labeled offline eval set — the contract the service must not regress on.
Each case: a question, whether it should be answered (grounded) or abstained,
and gold chunk ids that MUST appear among the citations when grounded.
"""
EVAL = [
    {"q": "Why is my shot bitter?",              "grounded": True,  "gold": ["d_grind#0", "d_taste#1"]},
    {"q": "What does a sour shot mean?",         "grounded": True,  "gold": ["d_taste#0"]},
    {"q": "What causes no crema?",               "grounded": True,  "gold": ["d_crema#0", "d_crema#1"]},
    {"q": "How do I fix a leaking portafilter?", "grounded": True,  "gold": ["d_leak#0"]},
    {"q": "How often should I clean the machine?","grounded": True, "gold": ["d_clean#0"]},
    # multi-hop bridge: 'reference machine' -> Silvia Pro -> warm-up/boiler
    {"q": "How long until my reference machine is ready to brew?",
                                                 "grounded": True,  "gold": ["d_temp#1", "d_warm#0", "d_grind#2"]},
    # out-of-domain: must abstain
    {"q": "Is the moon made of cheese?",         "grounded": False, "gold": []},
    {"q": "What is the capital of France?",      "grounded": False, "gold": []},
]
'''
FILES['tests/test_eval.py'] = r'''"""CI eval gate. Fails the build if quality drops below thresholds.
Run:  python -m pytest -q   (or)   python tests/test_eval.py
"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from rag_service import build_index, answer
from tests.eval_set import EVAL

GROUNDED_ACC_MIN = 0.85   # correct grounded/abstain decision
CITATION_HIT_MIN = 0.80   # for grounded cases, at least one gold chunk cited

def run_eval():
    idx = build_index()
    decisions, cite_hits, cite_total = 0, 0, 0
    rows = []
    for case in EVAL:
        a = answer(case["q"], idx)
        decision_ok = (a.grounded == case["grounded"])
        decisions += decision_ok
        cite_ok = None
        if case["grounded"]:
            cite_total += 1
            cite_ok = any(g in a.citations for g in case["gold"])
            cite_hits += cite_ok
        rows.append((case["q"], case["grounded"], a.grounded, a.hops, cite_ok, a.citations))
    dec_acc = decisions / len(EVAL)
    cite_acc = cite_hits / cite_total if cite_total else 1.0
    return dec_acc, cite_acc, rows

def test_quality_gate():
    dec_acc, cite_acc, rows = run_eval()
    assert dec_acc >= GROUNDED_ACC_MIN, f"decision acc {dec_acc:.2f} < {GROUNDED_ACC_MIN}"
    assert cite_acc >= CITATION_HIT_MIN, f"citation hit {cite_acc:.2f} < {CITATION_HIT_MIN}"

if __name__ == "__main__":
    dec_acc, cite_acc, rows = run_eval()
    for q, exp, got, hops, cite_ok, cites in rows:
        print(f"[{'ok' if exp==got else 'XX'}] grounded={got!s:5} hops={hops} "
              f"cite={cite_ok} :: {q[:44]}")
    print(f"\ndecision_acc={dec_acc:.2f}  citation_hit={cite_acc:.2f}")
    ok = dec_acc >= GROUNDED_ACC_MIN and cite_acc >= CITATION_HIT_MIN
    print("GATE:", "PASS" if ok else "FAIL")
    sys.exit(0 if ok else 1)
'''
FILES['pyproject.toml'] = r'''[project]
name = "rag-service"
version = "0.1.0"
description = "A compact, production-shaped RAG service (Phase 10 capstone)."
requires-python = ">=3.10"
dependencies = ["fastapi>=0.110", "uvicorn>=0.29", "pydantic>=2"]

[project.optional-dependencies]
dev = ["pytest>=8", "httpx>=0.27"]

[tool.pytest.ini_options]
addopts = "-q"
testpaths = ["tests"]
'''
FILES['Dockerfile'] = r'''FROM python:3.11-slim
WORKDIR /app
COPY pyproject.toml .
RUN pip install --no-cache-dir fastapi uvicorn "pydantic>=2"
COPY rag_service ./rag_service
EXPOSE 8000
CMD ["uvicorn", "rag_service.api:app", "--host", "0.0.0.0", "--port", "8000"]
'''
FILES['.github/workflows/ci.yml'] = r'''name: ci
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -e ".[dev]"
      # The eval gate runs with NO API key -> deterministic mock -> reproducible CI.
      - run: python -m pytest -q
'''
FILES['README.md'] = r'''# rag-service

A compact, production-shaped Retrieval-Augmented Generation service — the
capstone of a from-scratch RAG curriculum. It folds an entire retrieval stack
into one importable package and a single FastAPI `/ask` endpoint:

`chunk -> transform query -> hybrid retrieve + RRF -> agentic multi-hop
(self-routing) -> rerank -> grounded, cited, abstaining answer.`

## Why it's interesting
- **Runs with zero credentials.** A deterministic mock LLM stands in when no
  API key is set, so the service and its CI eval gate are fully reproducible.
- **Grounded or silent.** Answers cite their evidence `[chunk_id]` and abstain
  when the evidence is insufficient — no confident hallucinations.
- **Agentic when it must be, cheap when it can.** Easy questions terminate in
  one hop; bridge questions hop across documents via mined linking terms.
- **Quality is a build gate.** `tests/test_eval.py` asserts decision accuracy
  and citation hit-rate; a regression fails CI.

## Run it
```bash
pip install -e ".[dev]"
uvicorn rag_service.api:app --reload
curl -s localhost:8000/ask -H 'content-type: application/json' \
  -d '{"question":"Why is my shot bitter?"}'
python -m pytest -q          # the eval gate
```

Set `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` to swap the mock for a real model.
'''

for rel, text in FILES.items():
    path = os.path.join(BASE, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(text)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

print('Wrote', len(FILES), 'files under', BASE)
for rel in FILES:
    print('  ', rel)

## §3 · Chunking (L84) — documents in, retrievable units out

The knowledge base is a tiny home-espresso corpus, deliberately arranged so **some facts live one hop apart**: e.g. *"the reference machine"* is named only in the grind doc (it's the *Silvia Pro*), while its warm-up behaviour lives in a different doc. That shape is what forces the agentic loop later.

`chunk_docs()` cuts each document into overlapping sentence windows so no single fact gets split across a boundary. Chunk ids are deterministic (`<doc>#<n>`) — which is exactly what we cite.

In [ ]:
from rag_service.corpus import chunk_docs, DOCS
chunks = chunk_docs()
print(f"{len(DOCS)} docs -> {len(chunks)} chunks\n")
for c in chunks[:6]:
    print(f"{c.id:12} {c.text[:66]}")
# 💡 EXPERIMENT: try window=3, overlap=2 above and watch chunk count & overlap change.

## §4 · Hybrid retrieval + RRF (L85)

Two dependency-free signals over the same chunks:
- **Sparse** — IDF-weighted bag-of-words cosine (nails exact term overlap).
- **Dense** — character-trigram cosine, a transparent stand-in for embeddings (robust to word-form mismatch, no model download).

They disagree in useful ways, so we merge their *ranked lists* with **Reciprocal Rank Fusion** (`k=60`) — scale-agnostic, no score calibration needed. The same `fuse_rankings()` later merges the query variants from `transform`.

In [ ]:
from rag_service.retriever import HybridIndex
index = HybridIndex(chunks)
for c, s in index.search("bitter over extracted", k=4):
    print(f"{s:.4f}  {c.id:12} {c.text[:56]}")
# 💡 EXPERIMENT: search 'no foam' (everyday) vs 'crema' (technical) — see the vocab gap L88 fixes.

## §5 · Query transformation (L88)

Users type everyday words (*"flat"*, *"no foam"*); the corpus speaks technical (*"stale beans"*, *"crema"*, *"under-extracted"*). **Multi-Query** asks the LLM to paraphrase into corpus vocabulary, we retrieve for the original *and* each variant, then RRF-fuse. Keyless, the mock supplies a domain synonym expansion so this still closes part of the gap.

In [ ]:
from rag_service.transform import expand_query, multi_query_retrieve
print("variants:", expand_query("why is my shot flat with no foam?"))
hits, variants = multi_query_retrieve(index, "why is my shot flat with no foam?")
print("\nfused top hits:", [c.id for c, _ in hits])
# 💡 EXPERIMENT: set an ANTHROPIC_API_KEY in Colab Secrets and re-run — the rewrites get richer.

## §6 · Cross-encoder-style reranking (L86)

The retriever scores query and chunk *independently*. A cross-encoder looks at the **pair together** and is far more precise. We emulate that with a transparent pairwise scorer — query-term coverage plus proximity — that reorders the retriever's shortlist so the most on-point chunk floats to the top.

In [ ]:
from rag_service.rerank import rerank
shortlist = index.search("what does a leak from the portafilter mean", k=5)
print("retriever order:", [c.id for c, _ in shortlist])
print("reranked      :", [c.id for c, _ in rerank("leak from the portafilter", shortlist, top_n=3)])

## §7 · Grounding, citations & abstention (L87)

Three rules the answer step obeys: **(1)** answer only from retrieved evidence; **(2)** every claim carries a `[chunk_id]` citation; **(3)** if the evidence doesn't cover the question, **abstain** instead of guessing. `compose_answer()` builds the cited string; the mock grounds on the top evidence sentence, a real model writes fuller prose from the same evidence block.

In [ ]:
from rag_service.grounding import coverage, compose_answer
from rag_service.llm import call_llm
ev = [c for c, _ in rerank("bitter shot", index.search("bitter shot", k=5), top_n=3)]
cov, missing = coverage("why is my shot bitter", ev)
text, cites = compose_answer("why is my shot bitter", ev, call_llm)
print(f"coverage={cov:.2f}  citations={cites}\n{text}")

## §8 · The agentic loop + self-routing (L89)

This is the brain. Each hop: **retrieve → grade the evidence → if it answers, stop** (that's self-routing — easy questions terminate in one hop). If it doesn't, **mine bridge terms** (specific, recurring words in the current evidence that aren't in the query — the *links*) and **hop again** with an augmented query, up to a hop budget.

Watch the two traces below. The direct question grounds in **1 hop**. The bridge question — *"how long until my reference machine is ready to brew?"* — shares no words with the warm-up evidence, so hop 1 only finds *"the reference machine is the Silvia Pro"*; the loop mines **`silvia`/`pro`**, hops, and lands on the Silvia Pro's boiler/warm-up passage in **hop 2**.

In [ ]:
from rag_service import answer, build_index, llm_mode
idx = build_index()
print("LLM mode:", llm_mode(), "\n")

for q in ["Why is my shot bitter?",
          "How long until my reference machine is ready to brew?"]:
    a = answer(q, idx)
    print("Q:", q)
    for t in a.trace:
        print(f"   hop {t['hop']}: cov={t['coverage']} best={t['best_score']} "
              f"ev={t['evidence']}  q='{t['query'][:60]}'")
    print(f"   -> grounded={a.grounded} hops={a.hops} cites={a.citations}")
    print(f"   -> {a.text}\n")
# 💡 EXPERIMENT: raise grade_threshold to 0.95 in answer(...) and watch more questions hop or abstain.

## §9 · Serve it: the FastAPI `/ask` endpoint

The API is deliberately thin — validation, a single index built at startup, and a JSON response. All the intelligence is in `answer()`. We exercise it with FastAPI's `TestClient` (no server needed). If FastAPI isn't installed for some reason, we fall back to calling `answer()` directly so the cell still demonstrates the same contract.

In [ ]:
try:
    from fastapi.testclient import TestClient
    from rag_service.api import app
    client = TestClient(app)
    print("HEALTH:", client.get("/health").json())
    for q in ["Why is my shot bitter?", "Is the moon made of cheese?"]:
        r = client.post("/ask", json={"question": q}).json()
        print(f"\nPOST /ask  {q}")
        print("  grounded:", r["grounded"], "| hops:", r["hops"], "| cites:", r["citations"])
        print("  answer  :", r["answer"][:90])
except Exception as e:
    print("FastAPI path unavailable (%s) — using answer() directly:" % type(e).__name__)
    for q in ["Why is my shot bitter?", "Is the moon made of cheese?"]:
        a = answer(q, idx); print(q, "->", a.grounded, a.citations)

## §10 · Quality is a build gate, not a vibe

`tests/eval_set.py` is the contract: labeled questions, whether each should be **answered or abstained**, and the **gold chunks** that must appear in the citations. `tests/test_eval.py` computes two metrics and *fails CI* if either drops:

- **decision accuracy** — did we correctly ground vs abstain? (≥ 0.85)
- **citation hit-rate** — for grounded cases, is at least one gold chunk cited? (≥ 0.80)

Because the whole pipeline is deterministic under the mock, this gate gives the **same result on every machine and every CI run**. Run it:

In [ ]:
import subprocess, sys, os
res = subprocess.run([sys.executable, os.path.join(BASE, "tests", "test_eval.py")],
                     capture_output=True, text=True, cwd=BASE)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr); raise SystemExit("eval gate failed")

## §11 · Packaging, CI & Docker — what makes it *shippable*

These are already on disk. Together they're the difference between a gist and a project someone can adopt:

- **`pyproject.toml`** — `pip install -e ".[dev]"` and a console-free, standard build.
- **`.github/workflows/ci.yml`** — runs `pytest` (the eval gate) on every push/PR, **with no API key**, so contributors can't merge a quality regression.
- **`Dockerfile`** — `docker build` → `uvicorn rag_service.api:app` on port 8000.
- **`README.md`** — the 30-second pitch + run commands.

In [ ]:
for f in ["pyproject.toml", ".github/workflows/ci.yml", "Dockerfile"]:
    print("="*68, "\n#", f, "\n" + "="*68)
    print(open(os.path.join(BASE, f)).read())

## §12 · Ten production pitfalls (the ones that bite)

1. **Grading on query words, not evidence.** For *bridge* questions the answer shares no words with the question — a lexical grader will mis-say "grounded." Grade whether the evidence *answers*, and give the LLM the job in production.
2. **No abstention.** A service that always answers will confidently hallucinate on out-of-domain questions. Make "I don't know" a first-class output (and test it).
3. **Unbounded hops.** Always cap the agentic loop; a runaway loop is a runaway bill and a latency spike.
4. **Rerank on the original query after a bridge hop.** The answer chunk matches the *augmented* query — rerank against the current query or the bridge chunk never surfaces.
5. **Non-deterministic CI.** If your eval calls a live model, the gate flakes. Keep a deterministic mock behind the model seam so CI is reproducible.
6. **Citations that don't resolve.** Cite stable chunk ids and make sure every id maps back to real text; a citation you can't click is theater.
7. **Chunking that splits facts.** Zero overlap severs a fact across a boundary and both halves retrieve poorly. Use sentence windows with overlap.
8. **Fusing scores instead of ranks.** Sparse and dense scores live on different scales; averaging them is meaningless. RRF fuses *ranks* and sidesteps calibration.
9. **Transforming every query.** Rewriting is latency and cost; route it — only expand when the naive query retrieves weakly.
10. **Index rebuilt per request.** Build once at startup (as `api.py` does); rebuilding per `/ask` throws away all your throughput.

## §13 · Verification — the whole capstone, checked end to end

Every claim in this notebook, re-checked in code. This prints `PASS` only if the files exist, the pipeline behaves (direct = grounded+cited in 1 hop; bridge = 2 hops; out-of-domain = abstains), the HTTP layer answers, and the CI eval gate passes.

In [ ]:
import os, subprocess, sys
checks = []
def chk(name, cond): checks.append((name, bool(cond))); print(("PASS" if cond else "FAIL"), "-", name)

from rag_service import answer, build_index
idx = build_index()

# 1. repo materialized
need = ["rag_service/pipeline.py","rag_service/api.py","tests/test_eval.py",
        "pyproject.toml","Dockerfile",".github/workflows/ci.yml","README.md"]
chk("all project files written", all(os.path.exists(os.path.join(BASE,f)) for f in need))

# 2. direct question: grounded, cited, single hop (self-routing)
a = answer("Why is my shot bitter?", idx)
chk("direct Q grounded with a citation", a.grounded and len(a.citations) >= 1)
chk("direct Q self-routes in 1 hop", a.hops == 1)

# 3. bridge question: multi-hop, grounded, cites a Silvia-Pro chunk
b = answer("How long until my reference machine is ready to brew?", idx)
chk("bridge Q takes >=2 hops", b.hops >= 2)
chk("bridge Q grounded & cites boiler/warm-up evidence",
    b.grounded and any(c in b.citations for c in ["d_temp#1","d_temp#0","d_warm#0"]))

# 4. out-of-domain abstains (no hallucination)
o = answer("Is the moon made of cheese?", idx)
chk("out-of-domain question abstains", (not o.grounded) and o.citations == [])

# 5. every grounded answer carries a bracketed citation
chk("grounded answers embed [chunk_id]", "[" in a.text and "[" in b.text)

# 6. HTTP layer responds (graceful skip if FastAPI missing)
try:
    from fastapi.testclient import TestClient
    from rag_service.api import app
    r = TestClient(app).post("/ask", json={"question":"What causes no crema?"}).json()
    chk("POST /ask returns a grounded JSON answer", r["grounded"] and r["citations"])
except Exception as e:
    chk("POST /ask (skipped: %s)" % type(e).__name__, True)

# 7. CI eval gate passes
res = subprocess.run([sys.executable, os.path.join(BASE,"tests","test_eval.py")],
                     capture_output=True, text=True, cwd=BASE)
chk("CI eval gate PASS", res.returncode == 0 and "PASS" in res.stdout)

print("\n" + ("ALL CHECKS PASSED ✅" if all(c for _,c in checks)
              else "SOME CHECKS FAILED ❌"))
assert all(c for _, c in checks)

## §14 · Recap & what's next

**You shipped a service, not a script.** One `answer()` call now folds chunking, query transformation, hybrid retrieval + RRF, an agentic self-routing multi-hop loop, cross-encoder reranking, and grounded/cited/abstaining generation — behind a FastAPI `/ask`, packaged, Dockerized, and guarded by a deterministic CI eval gate. **Phase 10 is complete.**

**Make it a real portfolio piece (this week):**
- Push `rag_service/` to a public repo; the green CI badge is the selling point.
- Swap the toy corpus for a domain you care about (your own docs, a wiki dump). Only `corpus.py` changes.
- Set a real API key as a GitHub Actions *secret* and add a second CI job that runs a *few* live cases (keep the keyless gate as the required check).

**Where the curriculum goes next — Phase 11: Evaluation & Trust at Scale.** You've built agents (Phases 1–9) and a RAG service (Phase 10); the thing that separates a demo from a product is *proving* it stays good. Phase 11 plan (incremental, as always):
- **L91** — golden datasets & LLM-as-judge (turn today's 8-case gate into a real eval harness: faithfulness, context precision/recall, answer relevancy).
- **L92** — regression & canary evals in CI; scoring drift over time.
- **L93** — human-in-the-loop feedback capture → data flywheel.
- **L94** — red-teaming & safety evals (jailbreaks, prompt injection through retrieved docs).
- **L95** — cost/latency budgets and load-testing the `/ask` path.
- **L96** — Phase 11 capstone: a reusable `agent-evals` harness you can point at *any* of your services.

*(Autonomous scheduled run — no learner questions were pending, so the curriculum advanced L89 → L90 and Phase 11 was scoped as the natural next step. Ask anything and I'll fold it in.)*